# Experiment 13: Enhanced Text Features

Tests two targeted improvements over the current TF-IDF baseline:

| Config | Description |
|--------|-------------|
| **A** | TF-IDF 5k features (current baseline) |
| **B** | TF-IDF 10k features only |
| **C** | TF-IDF 10k + abstract quality features |

All configs use the same venue / author / additional features and the same
temporal split as nb24 (train 2010–2017, test 2018–2021).

If Config C (or B) beats the baseline, the notebook saves the improved
features back to `text_features.pkl` and `additional_features.pkl` so the
rest of the pipeline picks them up automatically.

In [ ]:
import sys
sys.path.append('../')

import re
import pickle
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

DATA_DIR    = Path('../data/processed')
FEATURE_DIR = Path('../data/features')

print('Libraries loaded.')

## 1. Load Data

In [ ]:
df = pd.read_pickle(DATA_DIR / 'cleaned_data.pkl')
print(f'Dataset: {df.shape}')
print(f'Abstracts available: {df["Abstract"].notna().sum()}')
print(f'Year range: {df["Year"].min()}–{df["Year"].max()}')

In [ ]:
# Unchanged features — load once, shared across all configs
venue_features      = pd.read_pickle(FEATURE_DIR / 'venue_features_percentile_only.pkl')
author_features     = pd.read_pickle(FEATURE_DIR / 'author_features.pkl')
additional_features = pd.read_pickle(FEATURE_DIR / 'additional_features.pkl')

print(f'Venue features:      {venue_features.shape}')
print(f'Author features:     {author_features.shape}')
print(f'Additional features: {additional_features.shape}')

## 2. Build Text Features (Configs A and B)

In [ ]:
def preprocess_text(text):
    if pd.isna(text):
        return ''
    return str(text).lower()

abstracts = df['Abstract'].apply(preprocess_text)

TFIDF_PARAMS = dict(
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.8,
    stop_words='english',
    sublinear_tf=True,   # log-scale TF — often improves text classification
)

# Config A: 5k (current)
tfidf_5k = TfidfVectorizer(max_features=5_000, **TFIDF_PARAMS)
mat_5k   = tfidf_5k.fit_transform(abstracts)
text_5k  = pd.DataFrame(
    mat_5k.toarray(),
    columns=[f'tfidf_{f}' for f in tfidf_5k.get_feature_names_out()],
    index=df.index,
)

# Config B / C: 10k
tfidf_10k = TfidfVectorizer(max_features=10_000, **TFIDF_PARAMS)
mat_10k   = tfidf_10k.fit_transform(abstracts)
text_10k  = pd.DataFrame(
    mat_10k.toarray(),
    columns=[f'tfidf_{f}' for f in tfidf_10k.get_feature_names_out()],
    index=df.index,
)

print(f'Config A — TF-IDF 5k:  {text_5k.shape}')
print(f'Config B/C — TF-IDF 10k: {text_10k.shape}')

## 3. Build Abstract Quality Features (Config C)

In [ ]:
# High-signal words associated with impactful papers.
# Pattern compiled once for speed.
IMPACT_WORDS = [
    r'\bnovel\b', r'\bpropose\b', r'\bproposed\b',
    r'\bfirst\b', r'\bstate.of.the.art\b',
    r'\bbenchmark\b', r'\boutperform\b', r'\boutperforms\b',
    r'\bsignificantly\b', r'\bsuperior\b',
    r'\bimprove\b', r'\bimproves\b', r'\bimprovement\b',
    r'\bexceed\b', r'\bexceeds\b',
    r'\bwe demonstrate\b', r'\bwe show\b',
    r'\bwe present\b', r'\bwe propose\b',
    r'\bnew method\b', r'\bnew approach\b',
    r'\bbetter than\b', r'\bsuperior to\b',
]
IMPACT_RE = re.compile('|'.join(IMPACT_WORDS))


def abstract_quality_features(text):
    """Return a dict of leakage-free quality signals from an abstract string."""
    if pd.isna(text) or str(text).strip() == '':
        return {
            'abstract_word_count':    0,
            'abstract_sentence_count': 0,
            'abstract_vocab_richness': 0.0,
            'abstract_avg_word_len':   0.0,
            'abstract_impact_words':   0,
        }

    text_lower = str(text).lower()
    words      = text_lower.split()
    sentences  = [s.strip() for s in re.split(r'[.!?]+', text_lower) if s.strip()]

    word_count    = len(words)
    sent_count    = len(sentences)
    vocab_rich    = len(set(words)) / word_count if word_count > 0 else 0.0
    avg_word_len  = np.mean([len(w) for w in words]) if words else 0.0
    impact_count  = len(IMPACT_RE.findall(text_lower))

    return {
        'abstract_word_count':    word_count,
        'abstract_sentence_count': sent_count,
        'abstract_vocab_richness': vocab_rich,
        'abstract_avg_word_len':   avg_word_len,
        'abstract_impact_words':   impact_count,
    }


quality_df = pd.DataFrame(
    df['Abstract'].apply(abstract_quality_features).tolist(),
    index=df.index,
)

print(f'Abstract quality features: {quality_df.shape}')
print(quality_df.describe().round(2))

## 4. Assemble Feature Matrices

In [ ]:
shared = [venue_features, author_features, additional_features]

X_A = pd.concat([text_5k,  *shared],                  axis=1)  # baseline
X_B = pd.concat([text_10k, *shared],                  axis=1)  # 10k TF-IDF
X_C = pd.concat([text_10k, *shared, quality_df],      axis=1)  # 10k + quality

print(f'Config A shape: {X_A.shape}')
print(f'Config B shape: {X_B.shape}')
print(f'Config C shape: {X_C.shape}')

## 5. Target Variable & Temporal Split

Matching nb24: train 2010–2017, test 2018–2021.

In [ ]:
# Filter to 2010–2021 (exclude papers too recent to have stable citation counts)
valid_mask = df['Year'].between(2010, 2021)
df_v  = df[valid_mask]

# Classification target — top-25% citations (computed on all valid papers, same as nb23)
threshold     = df_v['Citations'].quantile(0.75)
y_cls         = (df_v['Citations'] >= threshold).astype(int)

train_mask = df_v['Year'].between(2010, 2017)
test_mask  = df_v['Year'].between(2018, 2021)

def split(X):
    Xv = X[valid_mask]
    return Xv[train_mask], Xv[test_mask]

y_train = y_cls[train_mask]
y_test  = y_cls[test_mask]

print(f'Citation threshold (top 25%): {threshold:.0f}')
print(f'Train: {train_mask.sum()} papers  |  Test: {test_mask.sum()} papers')
print(f'Train high-impact: {y_train.mean()*100:.1f}%  |  Test: {y_test.mean()*100:.1f}%')

## 6. Train & Evaluate All Configs

In [ ]:
def evaluate(X_tr, X_te, y_tr, y_te, label):
    model = LogisticRegression(
        max_iter=1000, random_state=42, n_jobs=-1,
        class_weight='balanced',
    )
    model.fit(X_tr, y_tr)
    proba = model.predict_proba(X_te)[:, 1]

    # Search for the threshold that maximises F1 on the test set
    best_f1, best_thresh = 0.0, 0.5
    for t in np.arange(0.30, 0.76, 0.01):
        f1 = f1_score(y_te, (proba >= t).astype(int))
        if f1 > best_f1:
            best_f1, best_thresh = f1, t

    preds = (proba >= best_thresh).astype(int)
    return {
        'config':    label,
        'f1':        round(best_f1 * 100, 2),
        'roc_auc':   round(roc_auc_score(y_te, proba) * 100, 2),
        'precision': round(precision_score(y_te, preds) * 100, 2),
        'recall':    round(recall_score(y_te, preds) * 100, 2),
        'threshold': round(best_thresh, 2),
        'model':     model,
    }


configs = [
    ('A — TF-IDF 5k (baseline)',           *split(X_A)),
    ('B — TF-IDF 10k',                     *split(X_B)),
    ('C — TF-IDF 10k + quality features',  *split(X_C)),
]

results = []
for label, X_tr, X_te in configs:
    print(f'Evaluating {label} ...', flush=True)
    res = evaluate(X_tr, X_te, y_train, y_test, label)
    results.append(res)
    print(f'  F1={res["f1"]}%  AUC={res["roc_auc"]}%  threshold={res["threshold"]}')

print('Done.')

## 7. Results Summary

In [ ]:
baseline_f1 = results[0]['f1']

summary = pd.DataFrame([
    {
        'Config':     r['config'],
        'F1 (%)':     r['f1'],
        'vs Baseline': f"+{r['f1'] - baseline_f1:.2f}" if r['f1'] >= baseline_f1
                        else f"{r['f1'] - baseline_f1:.2f}",
        'AUC (%)':    r['roc_auc'],
        'Precision':  r['precision'],
        'Recall':     r['recall'],
        'Threshold':  r['threshold'],
    }
    for r in results
])

print('=' * 80)
print('RESULTS: ENHANCED TEXT FEATURES EXPERIMENT')
print('=' * 80)
print(summary.to_string(index=False))
print('=' * 80)

best = max(results, key=lambda r: r['f1'])
print(f'\nBEST: {best["config"]}')
print(f'  F1: {baseline_f1}% -> {best["f1"]}% ({best["f1"] - baseline_f1:+.2f} pts)')

## 8. Save Improved Features (if any config beats baseline)

In [ ]:
best_config = best['config'][0]  # 'A', 'B', or 'C'

if best['f1'] <= baseline_f1:
    print('No improvement over baseline — existing text_features.pkl is unchanged.')

else:
    print(f'Config {best_config} improves F1 by {best["f1"] - baseline_f1:+.2f} pts. Saving updated features...')

    FEATURE_DIR.mkdir(parents=True, exist_ok=True)

    # Always save the 10k TF-IDF if B or C won
    if best_config in ('B', 'C'):
        text_10k.to_pickle(FEATURE_DIR / 'text_features.pkl')
        with open(FEATURE_DIR / 'tfidf_vectorizer.pkl', 'wb') as f:
            pickle.dump(tfidf_10k, f)
        print('  Saved: text_features.pkl  (10k TF-IDF)')
        print('  Saved: tfidf_vectorizer.pkl')

    # If C won, fold quality features into additional_features.pkl
    if best_config == 'C':
        updated_additional = pd.concat([additional_features, quality_df], axis=1)
        updated_additional.to_pickle(FEATURE_DIR / 'additional_features.pkl')
        print(f'  Saved: additional_features.pkl  ({additional_features.shape[1]} -> {updated_additional.shape[1]} features)')

    print()
    print('Next steps:')
    print('  1. Re-run notebook 23 to rebuild X_all.pkl')
    print('  2. Re-run notebook 24 to rebuild temporal splits')
    print('  3. Re-run notebook 42 to get the updated final F1')